# MLP (Neural Network) classifier for regret

最关键的判断标准

重新定义 target 后，可能出现三种结果：

分类 AUC 高、回归 R² 低
Distance 可以作为 failure alarm，但不能精确估计 slowdown 大小。这完全可以形成一个可靠结论。
分类 AUC 和回归 R² 都高
Distance 既能检测 failure，也能估计严重程度，是最理想结果。
分类 AUC 接近 0.5，回归 R² 接近 0
MMD、KS、WS 无法单独判断 BAO failure，需要加入 Query 结构、baseline latency、cardinality、plan features 等信息。

## Import and Random seed

In [1]:
import random
import numpy as np
import pandas as pd

## 2. Data Load

In [4]:
# Read data from a CSV file into a pandas DataFrame
df = pd.read_pickle("dataset/tpc-ds/precessed_regret_queries_100_add9scores_scaled.pkl")

In [5]:
def generate_targets(df, threshold=1.1, psql_default=False):
    feature_columns = ["mmd_score", "ks_stat", "ks_log_pvalue", "ws_score",
                       "mmd","energy_distance","wasserstein","ks","js_divergence","js_distance","kl_x_to_y","kl_y_to_x","symmetric_kl"]
    # dataset["ks_log_pvalue"] = -np.log10(dataset["ks_pvalue"].clip(lower=1e-300))
    # dataset["reg_log"] = np.log10(dataset["regret"].clip(lower=1e-300))
    if psql_default:
        df["target"] = np.where((df["bao_latency"] > df["default_latency"]), 1, 0)
    else:
        df["target"] = np.where((df["reg_log"] > threshold), 1, 0)

    X = df[feature_columns]
    y = df["target"]
    return X, y

In [6]:
X, y = generate_targets(df, threshold=1.1)
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8101, 13)
y shape: (8101,)


In [7]:
X.columns

Index(['mmd_score', 'ks_stat', 'ks_log_pvalue', 'ws_score', 'mmd',
       'energy_distance', 'wasserstein', 'ks', 'js_divergence', 'js_distance',
       'kl_x_to_y', 'kl_y_to_x', 'symmetric_kl'],
      dtype='object')

## 2. Analize mlp prediction result

In [3]:
df = pd.read_csv("results/tpcds/mlp_binary_ray_smote/test_predictions.csv")
df.columns

Index(['y_true', 'prob_class_1', 'y_pred'], dtype='object')

In [9]:
TP = ((df['y_true'] == 1) & (df['y_pred'] == 1)).sum()
print(f"TP:{TP}")

TN = ((df['y_true'] == 0) & (df['y_pred'] == 0)).sum()
print(f"TN:{TN}")

FP = ((df['y_true'] == 0) & (df['y_pred'] == 1)).sum()
print(f"FP:{FP}")

FN = ((df['y_true'] == 1) & (df['y_pred'] == 0)).sum()
print(f"FN:{FN}")

TP:89
TN:958
FP:538
FN:36


In [8]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(df['y_true'], df['y_pred'])
print(cm)

[[958 538]
 [ 36  89]]
